# Relatório Mensal e Comparativo Institucional (por Ano)

Este notebook tem duas funções:
1. Carrega o arquivo mestre de um ano/órgão, agrupa por mês e salva o relatório CSV em uma subpasta (ex: `.../dados_viagens_2025/Relatorios_Mensais/`).
2. Carrega **todos** os relatórios dessa subpasta (ex: UFPB e UFCG daquele ano) e gera um dashboard comparativo **institucional** para o ano selecionado.

## 1. Importações e Configurações (Etapa 1)

In [66]:
import pandas as pd
import numpy as np
import altair as alt
import os
import glob
import re

# --- Configurações ---
ano = 2025 # Ano dos dados a serem processados e salvos
orgao = 'UFCG' # 'UFPB' ou 'UFCG'

# --- Caminho para o ARQUIVO DE ENTRADA (Master) ---
pasta_dados_ano = f'dadosViagens/dados_viagens{ano}'
arquivo_entrada = os.path.join(pasta_dados_ano, f'df_master_{orgao}_aereo_{ano}.csv') 

# --- Caminho para a PASTA DE SAÍDA (Relatórios Mensais) ---
PASTA_RELATORIOS_MENSAIS = os.path.join(pasta_dados_ano, 'Relatorios_Mensais')
os.makedirs(PASTA_RELATORIOS_MENSAIS, exist_ok=True)


# Colunas que serão usadas do arquivo de entrada
COL_ID_VIAGEM = 'Identificador do processo de viagem'
COL_DATA = 'Período - Data de início' 
COL_DISTANCIA = 'Distância (GCD)'
COL_EMISSOES = 'Emissões (KgCO2eq)'
COL_GASTOS_PASSAGENS = 'Valor passagens' 

print(f"CONFIGURAÇÃO: Processando {orgao} / {ano}")
print(f"-> Lendo de: {arquivo_entrada}")
print(f"-> Salvando em: {PASTA_RELATORIOS_MENSAIS}")

CONFIGURAÇÃO: Processando UFCG / 2025
-> Lendo de: dadosViagens/dados_viagens2025\df_master_UFCG_aereo_2025.csv
-> Salvando em: dadosViagens/dados_viagens2025\Relatorios_Mensais


## 2. Carregar e Preparar Dados (Etapa 1)

In [67]:
print(f"🔄 Carregando dados de: '{arquivo_entrada}'...")
df = pd.DataFrame() # Inicializa df vazio
try:
    df = pd.read_csv(arquivo_entrada)
    print(f"✅ Dados carregados com sucesso ({len(df)} viagens).")
    
    colunas_necessarias = [COL_ID_VIAGEM, COL_DATA, COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    colunas_faltantes = [col for col in colunas_necessarias if col not in df.columns]
    if colunas_faltantes:
        print(f"❌ ERRO: Colunas essenciais não encontradas: {colunas_faltantes}")
        df = pd.DataFrame() # Reseta df se faltar algo crítico
    else:
        print("✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.")

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{arquivo_entrada}' não encontrado.")
except Exception as e:
    print(f"❌ Ocorreu um erro ao carregar os dados: {e}")

# --- Processamento de Datas e Custos --- 
if not df.empty:
    print("🔄 Processando datas e custos...")
    df['Data_Viagem'] = pd.to_datetime(df[COL_DATA], format='%d/%m/%Y', errors='coerce')
    
    colunas_numericas_processar = [COL_DISTANCIA, COL_EMISSOES, COL_GASTOS_PASSAGENS]
    for col in colunas_numericas_processar:
         df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0)
    print("✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.")
    
    df.dropna(subset=['Data_Viagem'], inplace=True)
    
    df['Mes_Num'] = df['Data_Viagem'].dt.month 
    df['Mes_Str'] = df['Data_Viagem'].dt.strftime('%m') 
    df['Mes_Ano'] = f"{ano}-" + df['Mes_Str'] 
    
    print("✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).")
else:
    print("⚠️ DataFrame vazio, processamento pulado.")

🔄 Carregando dados de: 'dadosViagens/dados_viagens2025\df_master_UFCG_aereo_2025.csv'...
✅ Dados carregados com sucesso (320 viagens).
✅ Colunas necessárias (Data, Distância, Emissões, Passagens) encontradas.
🔄 Processando datas e custos...
✅ Colunas de Distância, Emissões e Passagens convertidas para numérico.
✅ Datas processadas e colunas de Mês criadas (ANO FORÇADO PELO CONFIG).


## 3. Agrupar por Mês e Salvar Relatório (Etapa 1)

In [68]:
df_mensal = pd.DataFrame() # Cria DF vazio por segurança

if not df.empty:
    print(f"🔄 Agrupando totais por Mês/Ano para {ano}...")
    
    # --- INÍCIO DA CORREÇÃO (GARANTIR 12 MESES) ---
    
    # 1. Agrupa os dados que existem
    df_agrupado = df.groupby(['Mes_Ano', 'Mes_Num']).agg(
        Total_Distancia_Km = (COL_DISTANCIA, 'sum'),
        Total_Emissoes_KgCO2eq = (COL_EMISSOES, 'sum'),
        Total_Viagens = (COL_ID_VIAGEM, 'count'),
        Total_Passagens = (COL_GASTOS_PASSAGENS, 'sum') 
    ).reset_index()

    # 2. Cria o template de 12 meses para o ano configurado
    meses_template = pd.DataFrame({'Mes_Num': range(1, 13)})
    meses_template['Mes_Ano'] = meses_template['Mes_Num'].apply(lambda x: f"{ano}-{x:02d}")

    # 3. Mescla o template com os dados agrupados
    df_mensal = pd.merge(
        meses_template, 
        df_agrupado, 
        on=['Mes_Num', 'Mes_Ano'], 
        how='left' # Mantém todos os 12 meses do template
    )

    # 4. Preenche os meses sem viagens com 0
    colunas_para_zerar = [
        'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq', 
        'Total_Viagens', 'Total_Passagens'
    ]
    df_mensal[colunas_para_zerar] = df_mensal[colunas_para_zerar].fillna(0)
    
    # 5. Garante os tipos corretos (especialmente para 'Total_Viagens')
    df_mensal['Total_Viagens'] = df_mensal['Total_Viagens'].astype(int)
    df_mensal['Mes_Num'] = df_mensal['Mes_Num'].astype(int)
    # --- FIM DA CORREÇÃO ---
    
    df_mensal.sort_values('Mes_Num', inplace=True)
    print("✅ Agrupamento concluído (garantido 12 meses).")
    
    print(f"\n--- Relatório Mensal: Totais de Viagens Aéreas ({orgao} {ano}) ---")
    colunas_exibir = ['Mes_Ano', 'Total_Viagens', 'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq', "Total_Passagens"]
    print(df_mensal[colunas_exibir].to_string(index=False, formatters={
        'Total_Distancia_Km': '{:,.0f} km'.format,
        'Total_Emissoes_KgCO2eq': '{:,.0f} kg'.format,
        'Total_Passagens': 'R$ {:,.2f}'.format 
    }))
    
    try:
        nome_arquivo_saida = f"relatorio_mensal_{orgao}_aereo_{ano}.csv"
        caminho_saida = os.path.join(PASTA_RELATORIOS_MENSAIS, nome_arquivo_saida)
        df_mensal.round(2).to_csv(caminho_saida, index=False)
        print(f"\n✅ Relatório mensal (com 12 meses) salvo com sucesso em: '{caminho_saida}'")
    except Exception as e_save:
        print(f"❌ Erro ao salvar relatório mensal: {e_save}")
        
else:
    print("⚠️ DataFrame vazio, agrupamento pulado.")

🔄 Agrupando totais por Mês/Ano para 2025...
✅ Agrupamento concluído (garantido 12 meses).

--- Relatório Mensal: Totais de Viagens Aéreas (UFCG 2025) ---
Mes_Ano  Total_Viagens Total_Distancia_Km Total_Emissoes_KgCO2eq Total_Passagens
2025-01              1           4,231 km                 847 kg     R$ 2,936.15
2025-02              7          32,717 km               6,252 kg    R$ 31,696.13
2025-03             10          37,852 km               7,220 kg    R$ 37,901.75
2025-04             18          57,196 km              10,787 kg    R$ 62,080.29
2025-05             18          74,251 km              14,133 kg    R$ 50,023.24
2025-06             28         107,720 km              20,436 kg   R$ 113,789.01
2025-07             48         332,246 km              65,517 kg   R$ 189,065.39
2025-08             16          69,624 km              13,526 kg    R$ 50,129.53
2025-09             32         146,929 km              28,893 kg   R$ 122,650.71
2025-10             62         265,9

## 4. Visualização (Gráfico do Ano Atual - Etapa 1)

In [69]:
if 'df_mensal' in locals() and not df_mensal.empty:
    print(f"🔄 Criando gráficos mensais para o ano {ano}...")
    
    base = alt.Chart(df_mensal).encode(
        x=alt.X('Mes_Ano', sort=alt.EncodingSortField(field="Mes_Num", op="min", order='ascending'), axis=alt.Axis(title='Mês')), 
        tooltip=['Mes_Ano',
                 alt.Tooltip('Total_Viagens', title='Nº de Viagens'),
                 alt.Tooltip('Total_Distancia_Km', title='Distância (km)', format=',.0f'),
                 alt.Tooltip('Total_Emissoes_KgCO2eq', title='Emissões (kg)', format=',.0f'),
                 alt.Tooltip('Total_Passagens', title='Custo Passagens (R$)', format=',.2f') 
                ]
    )
    
    bar_emissoes = base.mark_bar(color='#d9534f').encode(y=alt.Y('Total_Emissoes_KgCO2eq', axis=alt.Axis(title='Total Emissões (KgCO2eq)')))
    text_emissoes = base.mark_text(dy=15, color='black', baseline='top').encode(y=alt.Y('Total_Emissoes_KgCO2eq'), text=alt.Text('Total_Emissoes_KgCO2eq', format=',.0f'))
    chart_emissoes = (bar_emissoes + text_emissoes).properties(title='Emissões Totais por Mês', width=700)
    
    bar_distancia = base.mark_bar(color='#4682b4').encode(y=alt.Y('Total_Distancia_Km', axis=alt.Axis(title='Total Distância (Km)')))
    text_distancia = base.mark_text(dy=15, color='white', baseline='top').encode(y=alt.Y('Total_Distancia_Km'), text=alt.Text('Total_Distancia_Km', format=',.0f'))
    chart_distancia = (bar_distancia + text_distancia).properties(title='Distância Total por Mês', width=700)

    bar_passagens = base.mark_bar(color='#f0ad4e').encode(y=alt.Y('Total_Passagens', axis=alt.Axis(title='Total Passagens (R$)')))
    text_passagens = base.mark_text(dy=15, color='black', baseline='top').encode(y=alt.Y('Total_Passagens'), text=alt.Text('Total_Passagens', format=',.0f'))
    chart_passagens = (bar_passagens + text_passagens).properties(title='Custo Total de Passagens por Mês', width=700)
    
    line_viagens = base.mark_line(color='#5cb85c', point=True).encode(y=alt.Y('Total_Viagens', axis=alt.Axis(title='Nº de Viagens')))
    text_viagens = base.mark_text(dy=-10, color='black').encode(y=alt.Y('Total_Viagens'), text=alt.Text('Total_Viagens', format='d'))
    chart_viagens = (line_viagens + text_viagens).properties(title='Número de Viagens por Mês', width=700)

    dashboard_mensal = alt.vconcat(
        chart_emissoes, chart_distancia, chart_passagens, chart_viagens
    ).properties(
        title=f"Análise Mensal de Viagens Aéreas - {orgao} {ano}"
    ).resolve_scale(x='shared')
    
    try:
        pasta_dashboard_ano = os.path.join(pasta_dados_ano, 'dashboards_mensais')
        os.makedirs(pasta_dashboard_ano, exist_ok=True)
        arquivo_dashboard_html = os.path.join(pasta_dashboard_ano, f'dashboard_mensal_{orgao}_aereo_{ano}.html')
        dashboard_mensal.save(arquivo_dashboard_html)
        print(f"\n✅ Dashboard mensal DO ANO {ano} salvo como HTML em: '{arquivo_dashboard_html}'")
    except Exception as e_save_chart:
        print(f"❌ Erro ao salvar dashboard mensal como HTML: {e_save_chart}")

else:
    print("⚠️ DataFrame mensal vazio, gráficos do ano atual pulados.")

🔄 Criando gráficos mensais para o ano 2025...



✅ Dashboard mensal DO ANO 2025 salvo como HTML em: 'dadosViagens/dados_viagens2025\dashboards_mensais\dashboard_mensal_UFCG_aereo_2025.html'


In [70]:
dashboard_mensal # Descomente para ver o gráfico do ano atual

alt.VConcatChart(...)

---

## 5. Carregar Dados Comparativos (Etapa 2)

Esta etapa varre a subpasta `.../Relatorios_Mensais/` em busca de arquivos de **diferentes instituições** para o **ano atual**.

In [71]:
print(f"🔄 Carregando todos os relatórios da pasta: '{PASTA_RELATORIOS_MENSAIS}'...")

search_pattern = os.path.join(PASTA_RELATORIOS_MENSAIS, f"relatorio_mensal_*_aereo_{ano}.csv")
all_files = glob.glob(search_pattern)
all_files.sort()

all_data_list = []

if not all_files:
    print(f"❌ Nenhum arquivo encontrado com o padrão: {search_pattern}")
    print(f"   (Para ver um gráfico comparativo, processe e salve o relatório de outra instituição para o ano {ano})")
else:
    print(f"✅ Encontrados {len(all_files)} arquivos para comparação do ano {ano}:")
    for f in all_files:
        print(f"   - {os.path.basename(f)}")
        try:
            org_match = re.search(f'relatorio_mensal_(.*?)_aereo_{ano}\.csv$', os.path.basename(f))
            if not org_match:
                print(f"     ⚠️ Não foi possível extrair a instituição do nome do arquivo, pulando.")
                continue
                
            instituicao_arquivo = org_match.group(1)
            
            df_report = pd.read_csv(f)
            df_report['Instituicao'] = instituicao_arquivo
            all_data_list.append(df_report)
            
        except Exception as e_load_comp:
            print(f"     ❌ Erro ao carregar o arquivo {f}: {e_load_comp}")

if all_data_list:
    df_comparativo = pd.concat(all_data_list, ignore_index=True)
    print("\n✅ Todos os relatórios combinados em 'df_comparativo'.")
    print("\n--- Instituições Carregadas ---")
    print(df_comparativo['Instituicao'].value_counts().sort_index())
else:
    print("\n⚠️ Nenhum dado comparativo foi carregado.")
    df_comparativo = pd.DataFrame() # Cria vazio para evitar erros

🔄 Carregando todos os relatórios da pasta: 'dadosViagens/dados_viagens2025\Relatorios_Mensais'...
✅ Encontrados 2 arquivos para comparação do ano 2025:
   - relatorio_mensal_UFCG_aereo_2025.csv
   - relatorio_mensal_UFPB_aereo_2025.csv

✅ Todos os relatórios combinados em 'df_comparativo'.

--- Instituições Carregadas ---
Instituicao
UFCG    12
UFPB     9
Name: count, dtype: int64


## 6. Visualização Comparativa (Institucional por Ano - Etapa 2)

In [72]:
if 'df_comparativo' in locals() and not df_comparativo.empty:
    print("🔄 Criando dashboard comparativo institucional...")
    
    base_comp = alt.Chart(df_comparativo).encode(
        x=alt.X('Mes_Num:O', axis=alt.Axis(title='Mês', labels=True, ticks=True)),
        color=alt.Color('Instituicao:N', title='Instituição'),
        tooltip=['Instituicao', 'Mes_Ano', 
                 alt.Tooltip('Total_Emissoes_KgCO2eq', title='Emissões (kg)', format=',.0f'),
                 alt.Tooltip('Total_Distancia_Km', title='Distância (km)', format=',.0f'),
                 alt.Tooltip('Total_Passagens', title='Passagens (R$)', format=',.2f')
                ]
    ).properties(
        width=700
    )
    
    chart_comp_emissoes = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Emissoes_KgCO2eq', title='Total Emissões (KgCO2eq)')
    ).properties(title='Comparativo Institucional de Emissões')
    
    chart_comp_distancia = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Distancia_Km', title='Total Distância (Km)')
    ).properties(title='Comparativo Institucional de Distância')

    chart_comp_passagens = base_comp.mark_line(point=True).encode(
        y=alt.Y('Total_Passagens', title='Total Passagens (R$)')
    ).properties(title='Comparativo Institucional de Custo de Passagens')

    dashboard_comparativo = alt.vconcat(
        chart_comp_emissoes,
        chart_comp_distancia,
        chart_comp_passagens
    ).properties(
        title=f"Comparativo Institucional de Tendências Mensais - Ano {ano}"
    ).interactive()
    
    try:
        arquivo_dashboard_comp = os.path.join(PASTA_RELATORIOS_MENSAIS, f'dashboard_comparativo_institucional_{ano}.html')
        dashboard_comparativo.save(arquivo_dashboard_comp)
        print(f"\n✅ Dashboard COMPARATIVO salvo como HTML em: '{arquivo_dashboard_comp}'")
    except Exception as e_save_comp:
        print(f"❌ Erro ao salvar dashboard comparativo: {e_save_comp}")

else:
    print("⚠️ 'df_comparativo' vazio. Gráfico comparativo pulado.")


🔄 Criando dashboard comparativo institucional...

✅ Dashboard COMPARATIVO salvo como HTML em: 'dadosViagens/dados_viagens2025\Relatorios_Mensais\dashboard_comparativo_institucional_2025.html'


In [73]:
dashboard_comparativo # Descomente para ver o gráfico comparativo

alt.VConcatChart(...)